# Monetary Policy Transmission and Indian Asset Markets
## Goldman Sachs GIR-Style Quantitative Event Study (2013–2025)

**Author**: Avani Aravind  
**Framework**: Event Study, Market Model, Constant-Mean Yield Model, Policy Surprise OLS, Newey-West $\beta_2$ Sensitivity  

---

### Research Questions
1. How do RBI repo rate decisions transmit into Indian financial assets (**NIFTY 50, Bank NIFTY, 10Y G-Sec yield, USD/INR, individual bank stocks, NSE sectors**)?
2. Do rate hikes and cuts produce **asymmetric market reactions**?
3. Does the **surprise component** (deviation from consensus expectations) drive the magnitude of market reaction?
4. Which asset classes exhibit the highest **direct rate sensitivity ($\beta_2$)** net of broad market moves?


In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to system path
ROOT = Path('..')
sys.path.insert(0, str(ROOT.resolve()))

from src.rbi_events import get_full_calendar
from src.data_loader import download_all, compute_returns
from src.event_study import run_event_study, run_rate_sensitivity
from src.policy_surprise import run_surprise_suite
from src.statistical_tests import run_all_tests

print("Libraries and project modules successfully loaded.")

## 1. Load RBI Policy Calendar & Market Data
The RBI policy calendar contains 70 decisions (2013–2025), including scheduled and emergency meetings, policy stance, pre-MPC vs MPC era tags, and analyst poll expectations (post-2017).

In [ ]:
events = get_full_calendar()
print(f"Total RBI policy events loaded: {len(events)}")
print(events[["date", "decision_type", "change_bps", "new_rate", "cycle", "has_expectations"]].head(10))

## 2. Load Asset Prices and Compute Log Returns / Yield Differences
For equity and FX series, we compute daily log-returns  = \ln(P_t / P_{t-1})$.
For 10Y G-Sec yields (), we compute daily yield changes $\Delta y_t = y_t - y_{t-1}$ (log-returns are economically invalid for interest rate levels near zero).

In [ ]:
prices = download_all()
returns = compute_returns(prices)
print("Asset returns calculated. Data shape:", returns.shape)
returns.head()

## 3. Event-Study Analysis: AR & CAR Computation
- **Estimation Window**: -21$ trading days relative to meeting date $.
- **Market Model**: {i,\tau} = \alpha + \beta \cdot r_{\text{Nifty50},\tau} + \varepsilon_{\tau}$ OLS estimation for equity/FX assets.
- **Constant-Mean Yield Model**: {\text{yield},k} = \Delta y_k - \text{mean}(\Delta y_{\text{est}})$.
- **Event Windows**: $\{-1\}, \{0\}, \{0,+1\}, \{0,+3\}, \{0,+5\}$ trading days.

In [ ]:
study_results = run_event_study(returns, events)
car_panel = study_results["car_panel"]
ar_panel = study_results["ar_panel"]
print("CAR Panel shape:", car_panel.shape)
car_panel.head(10)

## 4. Rate Sensitivity ($\beta_2$) Estimation
Full-sample OLS with Newey-West HAC standard errors (maxlags=5):
28620r_{\text{asset},t} = \alpha + \beta_1 \cdot r_{\text{market},t} + \beta_2 \cdot \text{RateChange\_bps}_t + \varepsilon_t28620
Measures direct rate sensitivity net of broad market co-movement.

In [ ]:
sensitivity_df = run_rate_sensitivity(returns, events)
print(sensitivity_df)

## 5. Policy Surprise Decomposition
For meetings with expectations data (2017 onwards), we compute $\text{Surprise\_bps} = \text{Actual\_change} - \text{Expected\_change}$.
We regress CAR on Surprise\_bps using White HC3 robust standard errors.

In [ ]:
surprise_res = run_surprise_suite(car_panel, events)
print("Surprise Regressions (HC3 standard errors):")
print(surprise_res["regressions"])

## 6. Statistical Significance & Robustness Testing
1. One-sample t-tests ( = 0$)
2. Welch two-sample t-tests (Hike vs Cut CARs)
3. Kruskal-Wallis non-parametric regime tests (excluding COVID emergency events)

In [ ]:
test_suite = run_all_tests(car_panel, events)
print("=== Summary Statistics ===")
print(test_suite["summary_stats"].head(10))
print("
=== Hike vs Cut Welch T-Tests ===")
print(test_suite["hike_vs_cut"].head(10))

## 7. Conclusions & Research Findings
- **Transmission Direction**: Rate hikes produce negative abnormal returns on equities and positive yield changes on 10Y G-Secs, consistent with macro-finance theory.
- **Cross-Sectional Rate Sensitivity**: Banking sector and Realty demonstrate the highest $\beta_2$ sensitivities, while IT and FMCG are least rate-sensitive.
- **Surprise Factor**: Hawkish surprises drive negative equity CARs and positive yield ARs.